# CIMIS 5-station dataset (Sep 2021 to Sep 2026)

Pulls daily data plus hourly air temperature from the CIMIS API for stations
6 Davis, 70 Manteca, 71 Modesto, 39 Parlier, 5 Shafter, and returns one long
dataframe (one row per station per day) with chill and GDD columns.

Keep this notebook in the same folder as `pipeline.py`, `cimis_pull.py` and `agro.py`.
The first run makes roughly 150 API requests and caches the result to CSV; later runs read the CSV.

In [ ]:
import os, getpass, json
from pathlib import Path
import pandas as pd

import cimis_pull as cp
from pipeline import build_dataset, DEFAULT_STATIONS

app_key = os.environ.get("CIMIS_APP_KEY") or getpass.getpass("CIMIS app key: ")  # never hard-code it
CACHE = Path("data/cimis_5stations_daily_202109_202609.csv")

In [ ]:
if CACHE.exists():
    df = pd.read_csv(CACHE, parse_dates=["timestamp"], keep_default_na=False, na_values=[""])
    units = json.loads(CACHE.with_name(CACHE.stem + "_units.json").read_text())
else:
    df, units = build_dataset(
        cp.CimisClient(app_key),
        stations=DEFAULT_STATIONS,   # [6, 70, 71, 39, 5]
        gdd_bases=(50.0,),           # add crop-specific bases, e.g. (45.0, 50.0)
        chill_start="11-01",         # chill season reset (MM-DD)
        gdd_start="01-01",           # GDD season reset (MM-DD)
    )
    CACHE.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(CACHE, index=False)
    CACHE.with_name(CACHE.stem + "_units.json").write_text(json.dumps(units, indent=2))

df.shape

In [ ]:
df.head()

## Data quality at a glance
`record_missing` = CIMIS returned no daily record; `*_flag` = CIMIS QC flagged or missing;
`hourly_temp_count` < 24 means chill totals for that day are incomplete.

In [ ]:
df.groupby("station").agg(
    days=("timestamp", "size"),
    missing_days=("record_missing", "sum"),
    flagged_values=("n_flags", "sum"),
    short_hour_days=("hourly_temp_count", lambda s: (s < 24).sum()),
)

In [ ]:
# Season-end chill totals per station (chill season starts Nov 1; season label = start year)
df.groupby(["station", "chill_season"])[["chill_hours_cum", "chill_units_utah_cum", "chill_portions_cum"]].max().round(1)

## Feature columns
Measurement values, then QC codes/flags, then derived agronomic columns. `units` holds the unit of each measurement.

In [ ]:
id_cols = ["timestamp", "station", "station_name", "latitude", "longitude", "elevation_ft"]
qc_cols = [c for c in df.columns if c.endswith("_qc")]
flag_cols = [c for c in df.columns if c.endswith("_flag")] + ["record_missing", "n_flags"]
feature_cols = [c for c in df.columns if c not in id_cols + qc_cols + flag_cols]
print(len(feature_cols), "numeric features:", feature_cols)
units